# Personal Dataset Preprocessing Inspector

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

ROOT = '/content/drive/MyDrive/CW_Folder_UG'
sys.path.insert(0, f'{ROOT}/Code')

from utils import (
    AGE_LABELS,
    load_personal_dataset,
    read_rgb,
    match_dataset_resolution,
    crop_largest_face,
    preprocess_pipeline,
    _detect_face_candidates,
)
from skimage import color
from skimage.util import img_as_ubyte

In [ ]:
personal_paths, personal_labels = load_personal_dataset(ROOT, verbose=True)
y_personal = np.array(personal_labels)
print()
print('Paths loaded:', len(personal_paths))

## 1 - Raw image dimensions
Check how large the personal images are compared to typical dataset images (usually ≤ 400 px).

In [ ]:
print(f"{'File':<40} {'H':>6} {'W':>6} {'Longest':>8} {'Label':>12}")
print('-' * 78)
for path, label in zip(personal_paths, personal_labels):
    img = read_rgb(path)
    h, w = img.shape[:2]
    longest = max(h, w)
    flag = '  <-- oversized' if longest > 400 else ''
    print(f"{os.path.basename(str(path)):<40} {h:>6} {w:>6} {longest:>8}  {AGE_LABELS[label]:>12}{flag}")

## 2 - Face detection success rate

In [ ]:
RESOLUTION_MAX_SIDE = 400

def _face_detected(img):
    gray = img_as_ubyte(color.rgb2gray(img))
    min_dim = min(gray.shape[:2])
    min_face_size = max(24, int(min_dim * 0.06))
    return len(_detect_face_candidates(gray, min_face_size)) > 0

print(f"{'File':<40} {'Full res (used)':>16} {'Match-first (old)':>18}")
print('-' * 78)
full_hit, match_hit = 0, 0
for path, label in zip(personal_paths, personal_labels):
    img_full    = read_rgb(path)
    img_matched = match_dataset_resolution(img_full, max_side=RESOLUTION_MAX_SIDE)
    found_full  = _face_detected(img_full)
    found_match = _face_detected(img_matched)
    full_hit  += found_full
    match_hit += found_match
    name = os.path.basename(str(path))
    note = '  <-- would have failed' if found_full and not found_match else ''
    print(f"{name:<40} {'YES' if found_full else 'NO':>16} {'YES' if found_match else 'NO':>18}{note}")

n = len(personal_paths)
print()
print(f"Full-res (current pipeline): {full_hit}/{n}")
print(f"Match-first (old order):     {match_hit}/{n}")

## 3 - Per-image pipeline strip

In [ ]:
RESOLUTION_MAX_SIDE = 400
FINAL_SIZE = (128, 128)

for path, label in zip(personal_paths, personal_labels):
    name       = os.path.basename(str(path))
    true_label = AGE_LABELS[label]

    img_orig         = read_rgb(path)
    img_crop         = crop_largest_face(img_orig)
    img_crop_matched = match_dataset_resolution(img_crop, max_side=RESOLUTION_MAX_SIDE)
    img_final        = preprocess_pipeline(
        path,
        image_size=FINAL_SIZE,
        crop_face=True,
        match_resolution=True,
        resolution_max_side=RESOLUTION_MAX_SIDE,
    )

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f'{name}  |  label: {true_label}', fontsize=13, fontweight='bold')

    stages = [
        (img_orig,         f'1. Original\n{img_orig.shape[1]}x{img_orig.shape[0]}',               False),
        (img_crop,         f'2. Crop on full-res\n{img_crop.shape[1]}x{img_crop.shape[0]}',       False),
        (img_crop_matched, f'3. Crop + matched\n{img_crop_matched.shape[1]}x{img_crop_matched.shape[0]}', False),
        (img_final,        f'4. Final gray {FINAL_SIZE[1]}x{FINAL_SIZE[0]}',                      True),
    ]

    for ax, (img, title, is_gray) in zip(axes, stages):
        if is_gray:
            ax.imshow(img, cmap='gray', vmin=0, vmax=1)
        else:
            ax.imshow(np.clip(img, 0, 1))
        ax.set_title(title, fontsize=9)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

## 4 - Crop size consistency check

In [ ]:
RESOLUTION_MAX_SIDE = 400

print(f"{'File':<40} {'Crop H (orig)':>14} {'Crop W (orig)':>14} {'Aspect':>8} {'After match':>14} {'Note'}")
print('-' * 104)
for path in personal_paths:
    name = os.path.basename(str(path))
    img_orig     = read_rgb(path)
    crop_orig    = crop_largest_face(img_orig)
    crop_matched = match_dataset_resolution(crop_orig, max_side=RESOLUTION_MAX_SIDE)
    ch, cw = crop_orig.shape[:2]
    mh, mw = crop_matched.shape[:2]
    aspect = cw / ch
    note = '  <-- non-square (edge clip)' if abs(aspect - 1.0) > 0.15 else ''
    print(f"{name:<40} {ch:>14} {cw:>14} {aspect:>8.3f}  {mw}x{mh:<10}{note}")

## 5 - Quality comparison: personal vs dataset image

In [ ]:
from utils import load_coursework_dataset

train_paths, train_labels, _, _ = load_coursework_dataset(ROOT, verbose=False)

RESOLUTION_MAX_SIDE = 400
SAMPLE_PERSONAL_IDX = 0  # change to inspect a different personal image

ds_img       = read_rgb(train_paths[0])
pers_img     = read_rgb(personal_paths[SAMPLE_PERSONAL_IDX])
pers_crop    = crop_largest_face(pers_img)
pers_matched = match_dataset_resolution(pers_crop, max_side=RESOLUTION_MAX_SIDE)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(np.clip(ds_img, 0, 1))
axes[0].set_title(f'Dataset image\n{ds_img.shape[1]}x{ds_img.shape[0]} px  ({AGE_LABELS[train_labels[0]]})')
axes[0].axis('off')

axes[1].imshow(np.clip(pers_img, 0, 1))
axes[1].set_title(f'Personal - full res\n{pers_img.shape[1]}x{pers_img.shape[0]} px  ({AGE_LABELS[personal_labels[SAMPLE_PERSONAL_IDX]]})')
axes[1].axis('off')

axes[2].imshow(np.clip(pers_crop, 0, 1))
axes[2].set_title(f'Personal - crop on full-res\n{pers_crop.shape[1]}x{pers_crop.shape[0]} px')
axes[2].axis('off')

axes[3].imshow(np.clip(pers_matched, 0, 1))
axes[3].set_title(f'Personal - crop + matched\n{pers_matched.shape[1]}x{pers_matched.shape[0]} px  (max_side={RESOLUTION_MAX_SIDE})')
axes[3].axis('off')

plt.suptitle('Quality comparison: dataset vs personal', fontsize=13)
plt.tight_layout()
plt.show()

## 6 - Final preprocessed grid

In [ ]:
RESOLUTION_MAX_SIDE = 400
FINAL_SIZE = (128, 128)

n = len(personal_paths)
fig, axes = plt.subplots(2, n, figsize=(3 * n, 7))

for col, (path, label) in enumerate(zip(personal_paths, personal_labels)):
    name = os.path.basename(str(path))

    without_match = preprocess_pipeline(
        path, image_size=FINAL_SIZE, crop_face=True,
        match_resolution=False,
    )
    with_match = preprocess_pipeline(
        path, image_size=FINAL_SIZE, crop_face=True,
        match_resolution=True, resolution_max_side=RESOLUTION_MAX_SIDE,
    )

    axes[0, col].imshow(without_match, cmap='gray', vmin=0, vmax=1)
    axes[0, col].set_title(f'{name}\n{AGE_LABELS[label]}', fontsize=7)
    axes[0, col].axis('off')

    axes[1, col].imshow(with_match, cmap='gray', vmin=0, vmax=1)
    axes[1, col].set_title(f'{name}\n{AGE_LABELS[label]}', fontsize=7)
    axes[1, col].axis('off')

fig.text(0.005, 0.75, 'crop only', va='center', rotation='vertical', fontsize=10)
fig.text(0.005, 0.25, 'crop + match', va='center', rotation='vertical', fontsize=10)

plt.suptitle('All personal images - 128x128 grayscale (final model input)', fontsize=12)
plt.tight_layout()
plt.show()